# 01 — Очищення та хронологічні ознаки

## Мета

Перетворити синхронізовані MSFT + SPY ціни на файл, готовий до тренування.

**Хронологічна ознака** використовує лише інформацію, доступну на момент прогнозу: поточний або попередні торгові дні. Наприклад, `msft_return_5d` використовує ціни від `t-5` до `t`, а не майбутні ціни.

Конвенція: прогноз створюється **після закриття дня t** на результат наступних 5 торгових днів.


In [1]:
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = "https://github.com/Magnitol110/MarketLens.git"
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    repo_path = Path("/content/MarketLens")
    if not repo_path.exists():
        subprocess.run(["git", "clone", REPO_URL, str(repo_path)], check=True)
    os.chdir(repo_path)

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
assert (ROOT / "README.md").exists(), f"Repository root not found: {ROOT}"
print("Repository:", ROOT)
print("Running in Colab:", IN_COLAB)

Repository: D:\GitHub\Tenerefe Group Project\MarketLens
Running in Colab: False


In [2]:
import json
import sys

import numpy as np
import pandas as pd

sys.path.insert(0, str(ROOT))
from scripts.build_market_dataset import main as build_market_dataset
from scripts.build_training_features import main as build_training_features

raw_dir = ROOT / "data" / "raw"
raw_ready = (
    (raw_dir / "msft_kaggle_1986-03-13_2025-07-11.csv").exists()
    and bool(list(raw_dir.glob("msft_nasdaq_*.json")))
    and bool(list(raw_dir.glob("spy_nasdaq_*.json")))
)
if raw_ready:
    build_market_dataset()
else:
    print("Raw snapshots are absent; using committed processed price table")
training = build_training_features()

{
  "msft": {
    "rows": 10160,
    "min_date": "1986-03-13",
    "max_date": "2026-07-13"
  },
  "spy": {
    "rows": 2512,
    "min_date": "2016-07-13",
    "max_date": "2026-07-13"
  },
  "model_table": {
    "rows": 2512,
    "min_date": "2016-07-13",
    "max_date": "2026-07-13"
  }
}
{
  "input": "data/processed/msft_spy_daily.csv",
  "input_sha256": "c000ec0a70a11e04c77eac543fc5e2bdf388aca356340a6a25ecbd7829384797",
  "output": "data/processed/training_features.csv",
  "rows": 2477,
  "feature_count": 36,
  "first_date": "2016-08-10",
  "last_date": "2026-07-06",
  "target": "MSFT 5-trading-day return minus SPY 5-trading-day return",
  "class_threshold": 0.01,
  "horizon_trading_days": 5,
  "split_method": "chronological 70/15/15 with 5-row purge before validation and test",
  "splits": {
    "train": {
      "rows": 1735,
      "first_date": "2016-08-10",
      "last_date": "2023-07-03",
      "class_counts": {
        "neutral": 736,
        "outperform": 602,
        "underp

## Створені групи ознак

- дохідність за 1, 5, 10 і 20 торгових днів;
- відносна дохідність MSFT мінус SPY;
- положення ціни відносно ковзних середніх;
- волатильність за 5, 10 і 20 днів;
- відносний обсяг торгів;
- gap, внутрішньоденний рух і денний діапазон;
- циклічне кодування дня тижня та місяця.

Майбутня 5-денна дохідність використовується тільки для створення `target_class` і не потрапляє до ознак.


In [3]:
path = ROOT / "data" / "processed" / "training_features.csv"
training = pd.read_csv(path, parse_dates=["date"])
feature_columns = [column for column in training.columns if column not in {"date", "split", "target_class"}]

summary = training.groupby("split", sort=False).agg(
    rows=("date", "size"),
    first_date=("date", "min"),
    last_date=("date", "max"),
)
display(summary)
display(pd.crosstab(training["split"], training["target_class"]))
print("Features:", len(feature_columns))

 rows first_date  last_date
 1735 2016-08-10 2023-07-03
  368 2023-07-12 2024-12-24
  374 2025-01-03 2026-07-06
 neutral  outperform  underperform
     128         102           144
     736         602           397
     134         115           119
Features: 36


## Перевірки перед тренуванням

Поділ виконується хронологічно: приблизно 70% train, 15% validation, 15% test. П’ять рядків перед validation і test видаляються, бо їхня ціль використовувала б ціни з наступної частини.


In [4]:
assert training["date"].is_monotonic_increasing
assert not training["date"].duplicated().any()
assert training.isna().sum().sum() == 0
assert np.isfinite(training[feature_columns].to_numpy()).all()
assert set(training["split"]) == {"train", "validation", "test"}
assert set(training["target_class"]) == {"outperform", "neutral", "underperform"}

split_order = training.groupby("split")["date"].agg(["min", "max"])
assert split_order.loc["train", "max"] < split_order.loc["validation", "min"]
assert split_order.loc["validation", "max"] < split_order.loc["test", "min"]
print("QA passed: no nulls, duplicates, infinities, or chronological overlap")

QA passed: no nulls, duplicates, infinities, or chronological overlap


In [5]:
display(training.head(3))
display(training.tail(3))
print("Training-ready file:", path)
print("Rows:", len(training), "Features:", len(feature_columns))

      date split  msft_return_1d  msft_gap_return  msft_intraday_return  msft_range_pct  msft_return_5d  msft_ma_ratio_5d  msft_volatility_5d  msft_return_10d  msft_ma_ratio_10d  msft_volatility_10d  msft_return_20d  msft_ma_ratio_20d  msft_volatility_20d  msft_volume_ratio_20d  spy_return_1d  spy_gap_return  spy_intraday_return  spy_range_pct  spy_return_5d  spy_ma_ratio_5d  spy_volatility_5d  spy_return_10d  spy_ma_ratio_10d  spy_volatility_10d  spy_return_20d  spy_ma_ratio_20d  spy_volatility_20d  spy_volume_ratio_20d  excess_return_1d  excess_return_5d  excess_return_10d  excess_return_20d  day_of_week_sin  day_of_week_cos  month_sin  month_cos target_class
2016-08-10 train       -0.003093        -0.000687             -0.002407        0.008618        0.018431          0.001623            0.005100         0.032568           0.013184             0.004575         0.084283           0.031375             0.013332              -0.508801      -0.002475        0.000596            -0.003069

## Результат

`data/processed/training_features.csv` готовий до `02_baseline.ipynb`.

У baseline не можна перемішувати рядки або повторно робити випадковий split. Масштабування, якщо воно потрібне моделі, навчається тільки на `split == "train"`.
